# 5교시. Python 함수에 화면 붙이기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/05_gradio_basic.ipynb)

**목표:** Gradio 버튼과 mock 처리 함수를 연결합니다.

**결과물:** `app_05.py`

- 기본 경로는 API 키와 OCR 모델 다운로드가 필요 없습니다.
- 선택 실습은 기본값이 `False`입니다.
- 수업이 지정한 공개 실물·합성 샘플만 사용합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.metadata
import subprocess

required_gradio = "6.20.0"
try:
    installed_gradio = importlib.metadata.version("gradio")
except importlib.metadata.PackageNotFoundError:
    installed_gradio = None

if installed_gradio != required_gradio:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"gradio=={required_gradio}"]
    )

import gradio as gr
print("Gradio:", gr.__version__)


In [ ]:
SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'
SAMPLE_VLM_MARKDOWN = '# 샘플문구점\n\n거래일자: 2026-07-27\n\n| 품목 | 수량 | 단가 | 금액 |\n|---|---:|---:|---:|\n| 연필 | 2 | 1,000원 | 2,000원 |\n| 노트 | 1 | 3,000원 | 3,000원 |\n\n**합계: 5,000원**\n'
SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'mock'}


## 핵심 3개

1. 컴포넌트는 함수 입출력을 화면에 연결합니다.
2. 버튼 이벤트에는 함수·입력·출력이 필요합니다.
3. 화면과 처리 함수는 따로 확인합니다.


In [ ]:
def show_mock_result(file_path=None, processor="PaddleOCR"):
    status = f"MOCK {processor} 결과 — 업로드 문서를 읽지 않았습니다."
    source_text = (
        SAMPLE_VLM_MARKDOWN
        if processor == "PaddleOCR-VL"
        else SAMPLE_OCR_TEXT
    )
    return status, source_text, SAMPLE_RECEIPT


direct_result = show_mock_result()
assert "MOCK PaddleOCR 결과" in direct_result[0]
assert direct_result[2]["total_amount"] == 5000
print(direct_result[0])


## 실습. 버튼과 함수 연결


In [ ]:
with gr.Blocks(title="5교시 Document AI") as demo:
    gr.Markdown("# 영수증 Document AI · MOCK 실습")
    file_input = gr.File(label="합성 영수증", type="filepath")
    processor = gr.Radio(
        ["PaddleOCR", "PaddleOCR-VL"],
        value="PaddleOCR",
        label="문서 처리기",
    )
    process_button = gr.Button("mock 결과 보기", variant="primary")
    status = gr.Markdown()
    ocr_output = gr.Textbox(label="문서 인식 결과", lines=9)
    json_output = gr.JSON(label="구조화 JSON")

    process_button.click(
        fn=show_mock_result,
        inputs=[file_input, processor],
        outputs=[status, ocr_output, json_output],
    )

print("Gradio 화면 구성 완료")


In [ ]:
app_code = '''import gradio as gr

SAMPLE_TEXT = "합성 영수증 OCR 텍스트"
SAMPLE_JSON = {"source_mode": "mock"}

def show_mock_result(file_path=None, processor="PaddleOCR"):
    return f"MOCK {processor} 결과", SAMPLE_TEXT, SAMPLE_JSON

with gr.Blocks() as demo:
    file_input = gr.File(type="filepath")
    processor = gr.Radio(["PaddleOCR", "PaddleOCR-VL"], value="PaddleOCR")
    button = gr.Button("mock 결과 보기")
    status = gr.Markdown()
    text = gr.Textbox()
    data = gr.JSON()
    button.click(
        show_mock_result,
        [file_input, processor],
        [status, text, data],
    )

if __name__ == "__main__":
    demo.launch(share=False)
'''
output_path = OUTPUT_DIR / "app_05.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장 완료:", output_path)


In [ ]:
RUN_PUBLIC_DEMO = False

if RUN_PUBLIC_DEMO:
    demo.launch(share=True, max_file_size="5mb")
else:
    print("공개 공유 주소를 만들지 않았습니다. demo 객체 생성으로 검증했습니다.")


## mock 대체 경로

Gradio 화면이 열리지 않아도 `show_mock_result()` 직접 실행 결과가 같으면 완료입니다.
